In [ ]:
from google.colab import drive
from datasets import load_from_disk

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/Arabic_T5_Project/tokenized_data"

train_dataset = load_from_disk(f"{BASE_DIR}/train")
val_dataset = load_from_disk(f"{BASE_DIR}/validation")
test_dataset = load_from_disk(f"{BASE_DIR}/test")

print("=" * 60)
print("DATASETS RESTORED")
print("=" * 60)

print("TRAIN      :", len(train_dataset))
print("VALIDATION :", len(val_dataset))
print("TEST       :", len(test_dataset))

print()
print("COLUMNS:")
print(train_dataset.column_names)

Mounted at /content/drive
DATASETS RESTORED
TRAIN      : 189074
VALIDATION : 40516
TEST       : 40517

COLUMNS:
['input_ids', 'attention_mask', 'labels']


In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL_NAME = "bakrianoo/t5-arabic-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

tokenizer.add_special_tokens({
    "additional_special_tokens": [
        "<MAIN>",
        "<SUB>",
        "<TAGS>"
    ]
})

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False

model.to(device)

print("=" * 60)
print("MODEL + TOKENIZER READY")
print("=" * 60)

print("DEVICE :", device)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VOCAB  :", len(tokenizer))

print()
print("SPECIAL TOKENS:")

for token in ["<MAIN>", "<SUB>", "<TAGS>"]:
    print(token, "->", tokenizer.convert_tokens_to_ids(token))

tokenizer_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  847kB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.01GB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


MODEL + TOKENIZER READY
DEVICE : cpu
GPU    : CPU
VOCAB  : 35003

SPECIAL TOKENS:
<MAIN> -> 35000
<SUB> -> 35001
<TAGS> -> 35002


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding=True,
    return_tensors="pt"
)

print("=" * 60)
print("DATA COLLATOR READY")
print("=" * 60)
print("LABEL PAD TOKEN ID : -100")

DATA COLLATOR READY
LABEL PAD TOKEN ID : -100


In [ ]:
import os
import torch
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments

MODEL_NAME = "bakrianoo/t5-arabic-base"
OUTPUT_DIR = "/content/drive/MyDrive/Arabic_T5_Project/t5_training"
START_CHECKPOINT = os.path.join(OUTPUT_DIR, "checkpoint-1000")

BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 2
TOTAL_STEPS = 8865
START_STEP = 1000
REMAINING_STEPS = TOTAL_STEPS - START_STEP

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.isdir(START_CHECKPOINT):
    raise RuntimeError("checkpoint-1000 not found")

if not os.path.isfile(os.path.join(START_CHECKPOINT, "model.safetensors")):
    raise RuntimeError("model.safetensors not found")

print("=" * 70)
print("LOADING CHECKPOINT-1000")
print("=" * 70)
print("SOURCE :", START_CHECKPOINT)
print("START  :", START_STEP)
print("TOTAL  :", TOTAL_STEPS)
print("LEFT   :", REMAINING_STEPS)
print("=" * 70)

model = AutoModelForSeq2SeqLM.from_pretrained(
    START_CHECKPOINT,
    dtype=torch.float32
)

model = model.to(device)

model.config.use_cache = False

if hasattr(model, "generation_config"):
    model.generation_config.use_cache = False

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

torch.cuda.empty_cache()

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    max_steps=REMAINING_STEPS,

    learning_rate=5e-5,
    weight_decay=0.01,

    warmup_steps=0,

    optim="adamw_torch_fused",

    max_grad_norm=1.0,

    fp16=False,
    bf16=False,
    tf32=False,

    gradient_checkpointing=False,
    use_cache=False,

    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=None,
    save_only_model=False,

    report_to="none",

    remove_unused_columns=True,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    predict_with_generate=False,

    seed=42,
    data_seed=42,

    do_train=True,
    do_eval=True,

    skip_memory_metrics=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("=" * 70)
print("FINAL TRAINING")
print("=" * 70)
print("MODEL        :", MODEL_NAME)
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("START        : STEP 1000")
print("REMAINING    :", REMAINING_STEPS)
print("BATCH        :", BATCH_SIZE)
print("ACCUMULATION :", GRADIENT_ACCUMULATION)
print("EFFECTIVE    :", BATCH_SIZE * GRADIENT_ACCUMULATION)
print("FP16         : False")
print("BF16         : False")
print("TF32         : False")
print("LOGGING      : 100")
print("EVAL         : 1000")
print("SAVE         : 1000")
print("SAVE LIMIT   : NONE")
print("=" * 70)

if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free, _ = torch.cuda.mem_get_info()
    free /= 1024**3

    print("VRAM TOTAL   :", round(total, 2), "GB")
    print("VRAM FREE    :", round(free, 2), "GB")
    print("VRAM USED    :", round(total - free, 2), "GB")

print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)

trainer.train()

print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("MODEL SAVED TO:", OUTPUT_DIR)

LOADING CHECKPOINT-1000
SOURCE : /content/drive/MyDrive/Arabic_T5_Project/t5_training/checkpoint-1000
START  : 1000
TOTAL  : 8865
LEFT   : 7865


Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

FINAL TRAINING
MODEL        : bakrianoo/t5-arabic-base
GPU          : Tesla T4
START        : STEP 1000
REMAINING    : 7865
BATCH        : 32
ACCUMULATION : 2
EFFECTIVE    : 64
FP16         : False
BF16         : False
TF32         : False
LOGGING      : 100
EVAL         : 1000
SAVE         : 1000
SAVE LIMIT   : NONE
VRAM TOTAL   : 14.56 GB
VRAM FREE    : 7.8 GB
VRAM USED    : 6.76 GB
TRAINING STARTED


Step,Training Loss,Validation Loss
1000,33.726147,8.884335


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
1000,33.726147,8.884335
2000,18.064611,5.050978
3000,12.514164,3.353760


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import os
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

# ============================================================
# FINAL T5 TRAINING — CONTINUE FROM CHECKPOINT-3000
# ============================================================

MODEL_NAME = "bakrianoo/t5-arabic-base"

BASE_DIR = "/content/drive/MyDrive/Arabic_T5_Project/t5_training"

CHECKPOINT_DIR = os.path.join(
    BASE_DIR,
    "checkpoint-3000"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "final_model"
)

START_STEP = 3000
TOTAL_STEPS = 8865
REMAINING_STEPS = TOTAL_STEPS - START_STEP

BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 2
EFFECTIVE_BATCH = BATCH_SIZE * GRADIENT_ACCUMULATION

# ============================================================
# CHECKPOINT VERIFICATION
# ============================================================

print("=" * 70)
print("CHECKPOINT-3000 VERIFICATION")
print("=" * 70)

print("SOURCE :", CHECKPOINT_DIR)
print("START  :", START_STEP)
print("TOTAL  :", TOTAL_STEPS)
print("LEFT   :", REMAINING_STEPS)

if not os.path.isdir(CHECKPOINT_DIR):
    raise FileNotFoundError(
        f"\n❌ checkpoint-3000 not found:\n{CHECKPOINT_DIR}"
    )

MODEL_FILE = os.path.join(
    CHECKPOINT_DIR,
    "model.safetensors"
)

if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError(
        f"\n❌ model.safetensors not found:\n{MODEL_FILE}"
    )

print("MODEL FILE : FOUND ✅")
print("=" * 70)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# LOAD CHECKPOINT WEIGHTS ONLY
# ============================================================

print("LOADING CHECKPOINT-3000 WEIGHTS")
print("=" * 70)

model = AutoModelForSeq2SeqLM.from_pretrained(
    CHECKPOINT_DIR,
    dtype=torch.float32
)

model = model.to(device)

print("WEIGHTS LOADED ✅")

# ============================================================
# MODEL SETTINGS
# ============================================================

model.config.use_cache = False

if hasattr(model, "generation_config"):
    model.generation_config.use_cache = False

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

torch.cuda.empty_cache()

# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = Seq2SeqTrainingArguments(

    output_dir=BASE_DIR,

    # IMPORTANT:
    # We already loaded checkpoint-3000 weights.
    # Therefore train ONLY the remaining 5865 steps.
    max_steps=REMAINING_STEPS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    learning_rate=5e-5,
    weight_decay=0.01,

    # No old optimizer/scheduler state is loaded.
    warmup_steps=0,

    optim="adamw_torch_fused",

    max_grad_norm=1.0,

    # ========================================================
    # PRECISION — T4 SAFE
    # ========================================================

    fp16=False,
    bf16=False,
    tf32=False,

    # ========================================================
    # MEMORY
    # ========================================================

    gradient_checkpointing=False,
    use_cache=False,

    # ========================================================
    # LOGGING
    # ========================================================

    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    # ========================================================
    # VALIDATION
    # ========================================================

    eval_strategy="steps",
    eval_steps=1000,

    # ========================================================
    # CHECKPOINTS
    # ========================================================

    save_strategy="steps",
    save_steps=1000,

    # Keep ALL checkpoints.
    save_total_limit=None,

    save_only_model=False,

    # ========================================================
    # DATA
    # ========================================================

    remove_unused_columns=True,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # ========================================================
    # GENERATION
    # ========================================================

    predict_with_generate=False,

    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    seed=42,
    data_seed=42,

    # ========================================================
    # TRAIN / EVAL
    # ========================================================

    do_train=True,
    do_eval=True,

    report_to="none",

    skip_memory_metrics=True,
)

# ============================================================
# TRAINER
# ============================================================

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    processing_class=tokenizer,
)

# ============================================================
# FINAL CONFIG DISPLAY
# ============================================================

print("=" * 70)
print("FINAL T5 ARABIC TRAINING")
print("=" * 70)

print("MODEL        :", MODEL_NAME)

print(
    "DEVICE       :",
    device
)

print(
    "GPU          :",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("START        : CHECKPOINT-3000")
print("TOTAL        :", TOTAL_STEPS)
print("REMAINING    :", REMAINING_STEPS)

print("BATCH        :", BATCH_SIZE)
print("ACCUMULATION :", GRADIENT_ACCUMULATION)
print("EFFECTIVE    :", EFFECTIVE_BATCH)

print("EPOCHS       : 3")
print("MAX STEPS    :", REMAINING_STEPS)

print("FP16         : False")
print("BF16         : False")
print("TF32         : False")

print("LOGGING      : 100")
print("EVAL         : 1000")
print("SAVE         : 1000")
print("SAVE LIMIT   : NONE")

print("=" * 70)

# ============================================================
# VRAM
# ============================================================

if torch.cuda.is_available():

    total = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    free, _ = torch.cuda.mem_get_info()

    free /= 1024**3

    used = total - free

    print(
        "VRAM TOTAL   :",
        round(total, 2),
        "GB"
    )

    print(
        "VRAM FREE    :",
        round(free, 2),
        "GB"
    )

    print(
        "VRAM USED    :",
        round(used, 2),
        "GB"
    )

print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)

print(
    "STARTING FROM WEIGHTS: CHECKPOINT-3000"
)

print(
    "REMAINING STEPS:",
    REMAINING_STEPS
)

# ============================================================
# TRAIN
# ============================================================

trainer.train()

# ============================================================
# FINAL MODEL SAVE
# ============================================================

print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("=" * 70)
print("FINAL MODEL SAVED")
print("=" * 70)

print("PATH:", FINAL_DIR)

print("=" * 70)
print("DONE")
print("=" * 70)